In [ ]:
import json

from client import RagClient

# Set RAG_API_BASE_URL to this project's deployed API URL, including its stage.
# The client uses your default AWS credentials or the AWS_PROFILE environment variable.
rag_client = RagClient()


def upload_document(file_path):
    document_id = rag_client.upload(file_path)
    print(f"Uploaded {file_path}: {document_id}")
    return document_id

In [ ]:
files_to_upload = [
    "pdfs/Attention Is All You Need.pdf",
    "pdfs/Deep Residual Learning for Image Recognition.pdf",
    "pdfs/Generative Adversarial Networks.pdf",
    "pdfs/BERT Pre-training of Deep Bidirectional Transformers for Language Understanding.pdf",
    "pdfs/A Survey of Large Language Models.pdf",
]

document_ids = {}
for file_path in files_to_upload:
    document_ids[file_path] = upload_document(file_path)

for file_path, document_id in document_ids.items():
    print(f"Waiting for extraction and embeddings: {file_path}")
    document = rag_client.wait_document(document_id)
    print(f"Ready: {document['completed_chunks']} indexed chunks")

print(json.dumps(document_ids, indent=2))

In [ ]:
def test_full_rag_pipeline(question, doc_id=None):
    print(f"Question: {question}")
    retrieved = rag_client.query(question, doc_id)
    print(f"Retrieved {len(retrieved['top_chunks'])} passages")
    # Answer generation retrieves trusted context server-side and runs as a queued job.
    result = rag_client.answer(question, doc_id)
    print("\nANSWER:\n" + result["answer"])
    print("\nSOURCES:")
    for source in result["sources"]:
        print(
            f"[{source['source_id']}] {source['filename']} "
            f"(pages {source['pages']}, score {source['similarity_score']})"
        )
        print(source["excerpt"])
    return result

In [ ]:
MY_DOCUMENT_ID = document_ids.get("pdfs/Attention Is All You Need.pdf")
if not MY_DOCUMENT_ID:
    raise ValueError("Upload the Transformer paper successfully before running this cell.")

questions_to_ask = [
    "What is the methodology of this paper?",
    "What are the main conclusions?",
    "How does the Transformer use attention?",
    "What is a Use Case Diagram according to this document?",
]

for question in questions_to_ask:
    test_full_rag_pipeline(question, MY_DOCUMENT_ID)
    print("\n" + "=" * 60 + "\n")

Upload success is followed by document-status polling. Answers are queued and polled separately. Configure AWS access and deploy with the steps in `docs/DEPLOYMENT.md` before running this notebook. Saved outputs are intentionally cleared.